# Making a model smaller after it is trained

MichAl Academy, unit 4.10.

Run each cell with **Shift+Enter**.

A trained model is a pile of numbers. Quantization stores those numbers in fewer
bits. Nothing is retrained and nothing is redesigned; the same weights are simply
written down less precisely.

This notebook does it to a real model, **SmolLM2-135M-Instruct**, and separates
three things that usually get reported as one: how much smaller the file gets,
how much worse the model gets, and **which part of the rounding causes the
damage**. The third turns out to decide everything.


In [ ]:
import copy
import os
import statistics
import tempfile
import time
import warnings

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")


## What a weight costs

Every number has to be stored in some format. The default is **float32**: four
bytes each. The arithmetic below is the whole reason quantization exists, and the
right-hand column is why it is not optional at the sizes people actually deploy.


In [ ]:
for bits, name in ((32, "float32"), (16, "float16"), (8, "int8"), (4, "int4")):
    total = n_params * bits / 8
    print(f"{name:>8} ({bits:>2} bits): {total / 1024**2:>8.1f} MiB for this model, "
          f"{total * (70_000_000_000 / n_params) / 1024**3:>7.1f} GiB for a 70B model")


## The grid

int8 holds 256 whole numbers. Quantizing a layer means choosing a **scale**: one
float that says what one step of the grid is worth. Every weight is divided by
the scale, rounded to the nearest whole number, and stored as that integer.

The usual choice of scale is the largest magnitude in the layer divided by 127,
so that the biggest weight lands on the edge of the grid and nothing has to be
clipped. That choice is what the next cell inspects, on one real layer.


In [ ]:
layer = model.model.layers[0].mlp.gate_proj
w = layer.weight.data.flatten()

absmax = w.abs().max().item()
scale = absmax / 127
q = torch.clamp(torch.round(w / scale), -127, 127)
back = q * scale

print(f"layer            : model.layers[0].mlp.gate_proj, {w.numel():,} weights")
print(f"largest magnitude: {absmax:.6f}")
print(f"one grid step    : {scale:.8f}")
print(f"typical weight   : {w.abs().mean().item():.6f}")
print()
for i in range(5):
    print(f"  {w[i].item():+.6f}  ->  {int(q[i].item()):+4d}  ->  {back[i].item():+.6f}"
          f"   (off by {back[i].item() - w[i].item():+.6f})")
print()
print(f"grid values actually used: {int(q.unique().numel())} of 255")
print(f"weights above half the largest magnitude: "
      f"{(w.abs() > 0.5 * absmax).float().mean().item():.4%}")


## The outlier problem

One weight in that layer is about twenty-two times the typical magnitude, and it
alone sets the scale. Almost nothing else comes near it, so the ordinary weights
are crammed into the middle of the grid and most of the 255 available values are
never used.

The fix is to stop sharing one scale across the whole layer. **Per-channel**
quantization gives each output row its own scale, so a row without a large weight
gets a fine grid. It costs one extra float per row.


In [ ]:
W = layer.weight.data
typical = W.abs().mean().item()

s_tensor = W.abs().max() / 127
e_tensor = ((torch.clamp(torch.round(W / s_tensor), -127, 127) * s_tensor) - W).abs().mean().item()

s_chan = W.abs().amax(dim=1, keepdim=True) / 127
e_chan = ((torch.clamp(torch.round(W / s_chan), -127, 127) * s_chan) - W).abs().mean().item()

print(f"typical weight magnitude : {typical:.6f}")
print(f"per-tensor  average error: {e_tensor:.6f}  ({e_tensor / typical:.2%} of typical)")
print(f"per-channel average error: {e_chan:.6f}  ({e_chan / typical:.2%} of typical)")


## How much worse is the model

Error in the weights is not the thing that matters. What matters is what the
model then does, so the measure here is **cross-entropy loss** on ordinary text:
the model predicts each next token and is scored on how much probability it gave
the token that actually came. Lower is better.

Eight passages rather than one, because a single passage cannot tell a real
change from a lucky one. Four versions of the model are scored on exactly the
same text:

1. the original, float32
2. weights rounded to int8, one scale per output row
3. weights rounded to int8, one scale for the whole layer
4. PyTorch's dynamic quantization, which rounds the weights **and** the numbers
   flowing between layers


In [ ]:
PASSAGES = [
    "The bakery opens at six and the first tray of bread comes out a little after "
    "seven. By nine most of it has gone, and what is left goes half price at four "
    "in the afternoon. The owner has never advertised. She says the smell does the "
    "work, and the queue on a Saturday morning suggests she is right.",

    "A bicycle wheel stays upright for the same reason a spinning top does. Once it "
    "is turning, tipping it sideways produces a force at right angles to the push, "
    "and the wheel steers into the fall instead of collapsing under it. Riders learn "
    "this without being told, which is why explaining it rarely helps anyone balance.",

    "The train was delayed by forty minutes and nobody on the platform was told why. "
    "A guard eventually walked the length of the carriages saying there was a fault "
    "further up the line. Two passengers gave up and took a taxi. The rest waited, "
    "because the taxi rank was also empty.",

    "Sourdough is flour, water, salt and time. The starter is a colony of wild yeast "
    "and bacteria that has been fed often enough to stay alive, and it does the work "
    "that a packet of dried yeast would otherwise do in an hour. The trade is flavour "
    "against patience.",

    "The server refused the connection because the certificate had expired three days "
    "earlier. The team had known about the renewal date for some weeks, but nobody had "
    "been assigned to it, and the reminder went to a mailbox that nobody reads. When "
    "the alert finally arrived it was routed to the on-call engineer, who restarted "
    "the service twice before reading the log.",

    "Sea levels are measured against tide gauges fixed to the land, which means the "
    "land itself has to be accounted for. Parts of Scandinavia are still rising after "
    "the weight of the last ice sheet came off them, so a gauge there reads a fall "
    "where a satellite reads a rise. Both instruments are working correctly.",

    "She had played the same piece for eleven years and still practised the opening "
    "bars slowly every morning. The difficulty was never the notes. It was starting "
    "at the right speed, because the tempo chosen in the first two seconds decides "
    "whether the middle section is playable at all.",

    "The council planted four hundred trees along the ring road and lost about a fifth "
    "of them in the first summer. The survivors were mostly the ones near houses, "
    "where somebody had watered them. The following year the planting scheme came "
    "with a leaflet asking residents to do exactly that.",
]

n_tokens = sum(tok(p, return_tensors="pt").input_ids.shape[1] for p in PASSAGES)
print(f"{len(PASSAGES)} passages, {n_tokens} tokens")


@torch.no_grad()
def losses(m):
    out = []
    for p in PASSAGES:
        ids = tok(p, return_tensors="pt").input_ids
        logits = m(ids).logits[0, :-1]
        out.append(F.cross_entropy(logits, ids[0, 1:]).item())
    return out


def round_weights(m, per_channel):
    """int8 the weight matrices only, then put them back as float32.

    Storing the result as float32 is deliberate: it isolates the effect of the
    rounding from every other thing a real quantization library changes.
    """
    out = copy.deepcopy(m)
    for mod in out.modules():
        if isinstance(mod, torch.nn.Linear):
            Wm = mod.weight.data
            peak = Wm.abs().amax(dim=1, keepdim=True) if per_channel else Wm.abs().max()
            s = peak.clamp(min=1e-12) / 127
            mod.weight.data = torch.clamp(torch.round(Wm / s), -127, 127) * s
    out.eval()
    return out


qmodel = torch.quantization.quantize_dynamic(
    copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8)
qmodel.eval()

variants = {
    "float32": model,
    "int8 weights, per-channel": round_weights(model, True),
    "int8 weights, per-tensor": round_weights(model, False),
    "int8 weights and activations": qmodel,
}

scores = {name: losses(m) for name, m in variants.items()}
baseline = scores["float32"]

print()
print(f"{'':<30}{'mean loss':>11}{'vs float32':>12}{'worse on':>10}")
for name, L in scores.items():
    delta = [a - b for a, b in zip(L, baseline)]
    worse = sum(1 for d in delta if d > 0)
    shown = "" if name == "float32" else f"{statistics.mean(delta):+.4f}"
    count = "" if name == "float32" else f"{worse}/{len(L)}"
    print(f"{name:<30}{statistics.mean(L):>11.4f}{shown:>12}{count:>10}")


**Read the last column before the second one.** "Worse on 4 of 8 passages" is
what no effect looks like: the per-channel model is not slightly worse than
float32, it is indistinguishable from it, and the sign of the average is noise.
Per-tensor is worse on every passage, which is a small but real cost. Rounding
the activations as well is worse on every passage by an amount roughly fourteen
times larger.

So the damage is not in the weights. **Eight-bit weights are close to free; the
numbers flowing between the layers are what cannot be rounded carelessly.** That
is why the formats people actually deploy quantize weights only.


## Size on disk

Now the payoff. The saving is real but it is not the four times the bit
arithmetic promises, and the gap is worth understanding rather than explaining
away.


In [ ]:
def disk_size(m):
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        torch.save(m.state_dict(), f.name)
        size = os.path.getsize(f.name)
    os.unlink(f.name)
    return size


s_full, s_quant = disk_size(model), disk_size(qmodel)
print(f"float32 : {s_full / 1024**2:>7.1f} MiB")
print(f"int8    : {s_quant / 1024**2:>7.1f} MiB")
print(f"ratio   : {s_full / s_quant:>7.2f}x smaller")
print()

kept, rounded = 0, 0
for v in qmodel.state_dict().values():
    # Quantized linear layers arrive packed, as a tuple holding the int8 weight
    # and its bias, so a flat pass over the values would miss them entirely.
    for t in (v if isinstance(v, tuple) else (v,)):
        if not torch.is_tensor(t):
            continue
        if t.dtype == torch.qint8:
            rounded += t.numel()
        elif t.dtype == torch.float32:
            kept += t.numel()
print(f"stored as int8    : {rounded:>12,} numbers -> {rounded / 1024**2:>6.1f} MiB")
print(f"left at float32   : {kept:>12,} numbers -> {kept * 4 / 1024**2:>6.1f} MiB")
print(f"                                        {'-' * 6}")
print(f"accounted for     : {(rounded + kept * 4) / 1024**2:>26.1f} MiB "
      f"of the {s_quant / 1024**2:.1f} measured")
print()
print(f"embedding table      : {model.get_input_embeddings().weight.numel():>10,}")
print(f"normalisation weights: "
      f"{sum(p.numel() for n, p in model.named_parameters() if 'norm' in n):>10,}")


`quantize_dynamic` converts linear layers and nothing else, and the float32
remainder above is the embedding table plus the normalisation weights. The
embedding table is a lookup rather than a matrix multiply, so it is left alone,
and at 21% of the parameters it sets a floor under the file size on its own.

This model also **ties** its embedding table to its output layer: one set of
numbers used twice, once to turn a token into a vector and once to turn a vector
back into token scores. Only the second of those is a linear layer, so the saved
file ends up holding an int8 copy and a float32 copy of the same numbers.

Neither detail is worth memorising. The general shape is: **the saving applies to
the part you quantized, and everything else sets the floor.**


In [ ]:
@torch.no_grad()
def timed_generate(m, prompt, n=40, runs=3):
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    best = float("inf")
    out = None
    for _ in range(runs):
        t0 = time.time()
        out = m.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
        best = min(best, time.time() - t0)
    return best, tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


Q = "Explain in two sentences why a certificate expires."
t_full, out_full = timed_generate(model, Q)
t_quant, out_quant = timed_generate(qmodel, Q)
print(f"float32 : {t_full:.2f}s")
print(f"int8    : {t_quant:.2f}s  ({t_full / t_quant:.2f}x)")
print()
print("float32 says:", repr(out_full[:110]))
print("int8 says   :", repr(out_quant[:110]))


**Do not write that speed-up down as a fact about int8.** Re-run the cell and it
moves: five runs of this notebook on one machine ranged from 1.71x to 3.11x,
because the timing shares a CPU with everything else on it. Size and loss
came out identical to four decimal places every time. Of the three numbers
quantization is sold on, the one quoted most often is the one that reproduces
least.


## The other way to get a smaller model

Quantization keeps the model and writes it down in fewer bits. **Distillation**
throws the model away and trains a new, smaller one on its output.

The idea is that a trained model's output says more than the label does. The
label says "seven"; the model's output also says how much of a two it looked
like. That opinion about the wrong answers is what the small model is supposed to
learn from.

This section tests it on handwritten digits, which is small enough to train on a
CPU in under a minute. **It does not work here**, and the cell after it measures
why.


In [ ]:
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xd, yd, test_size=0.3, random_state=0, stratify=yd)
scaler = StandardScaler().fit(Xd_tr)
Xd_tr = torch.tensor(scaler.transform(Xd_tr), dtype=torch.float32)
Xd_te = torch.tensor(scaler.transform(Xd_te), dtype=torch.float32)
yd_tr, yd_te = torch.tensor(yd_tr), torch.tensor(yd_te)


def net(hidden, seed):
    """Seeded at construction, because the starting weights are the only random
    thing here: every step below is full-batch with no shuffling and no dropout,
    so two runs from the same start give the same answer to the last decimal."""
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(64, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, 10))


def fit(m, soft=None, T=4.0, alpha=0.0, epochs=800):
    """Train on the labels, optionally mixing in the teacher's distribution.

    alpha=0 is ordinary training. alpha=0.7 is distillation: seven parts
    teacher, three parts label. The T*T factor is the standard correction for
    the gradients shrinking as the temperature rises.
    """
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad()
        out = m(Xd_tr)
        loss = (1 - alpha) * F.cross_entropy(out, yd_tr)
        if alpha > 0:
            loss = loss + alpha * (T * T) * F.kl_div(
                F.log_softmax(out / T, 1), soft, reduction="batchmean")
        loss.backward()
        opt.step()
    return m


@torch.no_grad()
def test_acc(m):
    return (m(Xd_te).argmax(1) == yd_te).float().mean().item()


# One teacher per seed, reused by every student below. Retraining it inside the
# loop would quadruple the runtime and change nothing.
TEACHERS = [fit(net(256, seed)) for seed in range(5)]
LOGITS = [t(Xd_tr).detach() for t in TEACHERS]
print(f"teacher accuracy: {statistics.mean(test_acc(t) for t in TEACHERS):.4f}")
print()

print(f"{'':<10}{'student':>10}{'distilled':>11}{'distilled wins':>16}")
for hidden in (8, 16):
    for T in (2.0, 4.0):
        plain, distilled = [], []
        for seed in range(5):
            soft = F.softmax(LOGITS[seed] / T, 1)
            plain.append(test_acc(fit(net(hidden, seed + 100))))
            distilled.append(test_acc(
                fit(net(hidden, seed + 100), soft=soft, T=T, alpha=0.7)))
        wins = sum(1 for a, b in zip(distilled, plain) if a > b)
        print(f"h={hidden:<2} T={T:<4}{statistics.mean(plain):>10.4f}"
              f"{statistics.mean(distilled):>11.4f}{f'{wins}/5':>16}")


## Why it did not work

Distillation moves the teacher's opinion about the **wrong** answers. If the
teacher does not have one, there is nothing to move, and the cell below measures
exactly that.

Entropy is the measure: how spread out a distribution is, in nats. Zero means all
the probability on one answer. The most a distribution over ten classes can have
is the natural log of 10, about 2.30.


In [ ]:
with torch.no_grad():
    logits = LOGITS[0]
    for T in (1.0, 2.0, 4.0):
        p = F.softmax(logits / T, 1)
        ranked = p.sort(1, descending=True).values
        ent = -(p * p.clamp_min(1e-12).log()).sum(1).mean().item()
        print(f"T={T}: top answer {ranked[:, 0].mean():.4f}, "
              f"runner-up {ranked[:, 1].mean():.4f}, entropy {ent:.4f} nats")
print(f"a flat opinion over ten digits would be {torch.log(torch.tensor(10.0)):.4f} nats")


At temperature 1 the teacher gives its answer 0.9999 and the runner-up 0.0001.
Its distribution **is** the label, with decimal places. Raising the temperature
spreads out what is there, and what is there is nothing.

Digits is too easy for this. Distillation pays where a good model is still
genuinely uncertain in a structured way, which is the case on the language tasks
it is normally used for — and reading that condition off the teacher first is
cheaper than training a student to find out.


## What this unit measured

- **Quantization is a storage decision, not a training one.** Nothing was
  retrained. The same weights were written down in fewer bits.
- **Rounding weights to 8 bits is close to free**, provided each output row gets
  its own scale. Sharing one scale across a whole layer costs a little, because
  a single outlier weight stretches the grid for everything else.
- **Rounding the numbers between layers is not free**, and it caused essentially
  all of the quality loss measured here.
- **The saving is bounded by whatever you did not quantize.** Here that is the
  embedding table, and it is why the file shrank by about two times rather than
  four.
- **Distillation needs an uncertain teacher.** On a task where the teacher is
  certain, its output carries no more than the label and the student gains
  nothing.
